# UE02 - Korrelationsanalyse cont. und Regression: Modellanpassung

Nachdem wir uns in UE01 schon mit der grundlegenden Funktionalität der Bibliothek [matplotlib](https://matplotlib.org/) vertraut gemacht haben und die Korrelationsanalyse per [NumPy corrcoef()](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html) angeschnitten haben, werden wir uns in dieser Übung mit komplexeren Konzepten von matplotlib wie [Subplots](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/subplots_demo.html) und der Modellanpassung per Regression beschäftigten.

Eine komfortable Implementierung des Konzepts der linearen Regression unterschiedlicher Ordnungen bietet die NumPy-Klasse [Polynomial](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.Polynomial.html).
Weitere Infos zur Verwendung dieser Klasse könnt Ihr [dieser Übersichtsseite](https://numpy.org/doc/stable/reference/routines.polynomials.html) sowie der Dokumentation zur Methode [Polynomial.fit()](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.Polynomial.fit.html) entnehmen.

Zur Lösung der Problemstellungen und für weitere Erklärungen zu den in diesem Notebook verwendeten Formeln empfehle ich darüber hinaus das Skript zur Vorlesung Daten und Statistik von Dr. Stefan Wegenkittl.
Für diese Übung sind dabei insbesondere die Seiten 12-19 wichtig.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.polynomial import Polynomial

%matplotlib inline

## UE02.a)
Korrelationsanalyse der Merkmale des Datensatzes correlation.csv.
<ol>
    <li>Platziere die Datei <tt>correlation.csv</tt> aus dem Moodlekurs im Ordner <tt>data/</tt>. Lade anschließend <tt>data/correlation.csv</tt> in die Variable <tt>correlation</tt>.
        <br>
        <strong>Vorsicht</strong>: Sind Daten in Textformat (<tt>.txt</tt>, <tt>.csv</tt>, etc.) abgelegt, stellt NumPy eigene Funktionen (bspw. <a href="https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html">loadtxt()</a> oder <a href="https://numpy.org/doc/stable/reference/generated/numpy.genfromtxt.html">genfromtxt()</a> für das Laden des Inhalts in NumPy-Arrays bereit.
    </li>
    <li>Erstelle Scatterplots (oder eine Figure mit entsprechend vielen Subplots), die den ersten Spaltenvektor (X-Achse) gegen die restlichen Spaltenvektoren (Y-Achse) stellen, um damit die Korrelationen der Spalten zu untersuchen.</li>
    <li>Berechne die Korrelationskoeffizienten $r_{xy} = \cfrac{s_{xy}}{s_x \cdot s_y}$ des ersten Spaltenvektors in Bezug zu den restlichen Spalten in <tt>correlation</tt> und platziere diese Information formattiert im dazugehörigen Scatter-, bzw. Subplot.</li>
    <li>Nicht vergessen: entsprechende Beschriftungen für den/die Titel sowie die X-, und Y-Achse(n).</li>
    <li>Interpretiere das Ergebnis in Kommentarform oder als Markdownzelle.</li>
</ol>

In [ ]:
# datensatz laden — zeilen sind beobachtungen, spalten sind merkmale
correlation = np.genfromtxt('data02/correlation.csv', delimiter=',')

xCol = correlation[:, 0]        # spalte 0 kommt bei allen plots auf die x-achse
otherCols = correlation[:, 1:]  # alle anderen spalten kommen auf die y-achse
numSubplots = otherCols.shape[1]
n = len(xCol)

# 4x4 grid passt genau für alle 16 vergleiche
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i in range(numSubplots):
    yCol = otherCols[:, i]
    ax = axes[i]

    # pearson-korrelationskoeffizient: r_xy = s_xy / (s_x · s_y)
    # s_xy ist die empirische kovarianz — misst wie x und y gemeinsam variieren
    # durch division mit den standardabweichungen wird r auf [-1, 1] normiert
    sXY = np.sum((xCol - xCol.mean()) * (yCol - yCol.mean())) / (n - 1)
    sX = xCol.std(ddof=1)
    sY = yCol.std(ddof=1)
    corrCoef = sXY / (sX * sY)

    ax.scatter(xCol, yCol, s=10, alpha=0.6)
    ax.set_title(f'Col 0 vs Col {i + 1}\nr = {corrCoef:.4f}', fontsize=9)
    ax.set_xlabel('Column 0', fontsize=8)
    ax.set_ylabel(f'Column {i + 1}', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('UE02.a) – Korrelationsanalyse: Spalte 0 gegen alle anderen Spalten', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation UE02.a)**

Der Korrelationskoeffizient $r_{xy} \in [-1, 1]$ misst Stärke und Richtung des **linearen** Zusammenhangs zwischen zwei Merkmalen:

- $r \approx +1$: starker positiver linearer Zusammenhang (beide Merkmale steigen gemeinsam)
- $r \approx -1$: starker negativer linearer Zusammenhang (ein Merkmal steigt, das andere sinkt)
- $r \approx 0$: kein (oder kein *linearer*) Zusammenhang

Aus den Scatterplots ist erkennbar, dass die Spalten in zwei Gruppen zerfallen: einige zeigen einen klaren negativen Zusammenhang mit Spalte 0 (z.B. Col 8, 9, 10), andere einen positiven (Col 11–16). Spalten nahe der Mitte des Datensatzes (Col 6–7) zeigen kaum Korrelation — dort ist der r-Wert nahe null.

Wichtig: Ein hoher |r|-Wert belegt nur linearen Zusammenhang. Nichtlineare Muster werden von r nicht erfasst.

## UE02.b)
Erweiterung von UE02.a) um die lineare Regression
<ol>
    <li>Berechne für alle (Sub-)Plots aus UE02.a) den quadrierten Korrelationskoeffizienten $r_{xy}^{2} = \left(\cfrac{s_{xy}}{s_x \cdot s_y}\right)^2$.</li>
    <li>Errechne für all jene (Sub-)Plots, für die  $r_{xy}^{2} >= 0.5$ gilt, die lineare Regression erster Ordnung.</li>
    <li>Zeichne die lineare Regression erster Ordnung in die betreffenden (Sub-)Plots ein.</li>
    <li>Nicht vergessen: entsprechende Beschriftungen für den/die Titel sowie die X-, und Y-Achse(n).</li>
    <li>Interpretiere das Ergebnis in Kommentarform oder als Markdownzelle.</li>
</ol>

In [ ]:
# gleicher aufbau wie a), aber jetzt zeichnen wir wo r² >= 0.5 die lineare regression ein
# r² ist das bestimmtheitsmaß — gibt an welcher anteil der varianz in y durch das lineare modell erklärt wird
# unter 0.5 ist das modell nicht wirklich sinnvoll

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i in range(numSubplots):
    yCol = otherCols[:, i]
    ax = axes[i]

    sXY = np.sum((xCol - xCol.mean()) * (yCol - yCol.mean())) / (n - 1)
    sX = xCol.std(ddof=1)
    sY = yCol.std(ddof=1)
    corrCoef = sXY / (sX * sY)
    rSq = corrCoef ** 2

    ax.scatter(xCol, yCol, s=10, alpha=0.6, label='data')

    if rSq >= 0.5:
        # erste regressionsgerade y = a·x + b einzeichnen
        # Polynomial.fit minimiert die summe der quadratischen residuen (methode der kleinsten quadrate)
        linModel = Polynomial.fit(xCol, yCol, deg=1)
        xLine = np.linspace(xCol.min(), xCol.max(), 200)
        ax.plot(xLine, linModel(xLine), color='red', linewidth=1.5, label='lin. Regression')

    ax.set_title(f'Col 0 vs Col {i + 1}\nr = {corrCoef:.4f}, r² = {rSq:.4f}', fontsize=9)
    ax.set_xlabel('Column 0', fontsize=8)
    ax.set_ylabel(f'Column {i + 1}', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

plt.suptitle('UE02.b) – Lineare Regression für Spalten mit r² ≥ 0.5', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation UE02.b)**

Die lineare Regression (rote Linie) wird nur dort eingezeichnet, wo $r^2 \geq 0.5$ gilt — d.h. wo das lineare Modell mindestens 50 % der Varianz in y erklärt.

Spalten mit hohem $|r|$ (z.B. Col 8–10 mit stark negativem r, Col 11–16 mit positivem r) erhalten eine Regressionsgerade, die den Trend gut trifft. Spalten nahe Col 6–7 liegen unter der Schwelle — dort wäre eine Gerade irreführend, da kein linearer Zusammenhang vorliegt.

Der Vergleich zwischen a) und b) zeigt: nicht jede Korrelation rechtfertigt ein lineares Modell. Das Bestimmtheitsmaß $r^2$ ist dabei das entscheidende Filterkriterium.

## UE02.c)
Berechnung und Plotting linearer Regressionskurven unterschiedlicher Ordnung für die Daten in `regression.npy` mit anschließender visueller Analyse der Ergebnisse.

<ol>
    <li>Platziere die Datei <tt>regression.npy</tt> aus dem Moodlekurs im Ordner <tt>data/</tt>. Lade anschließend <tt>data/regression.npy</tt> in die Variable <tt>regression</tt>.</li>
    <li>Extrahiere die erste Spalte des Arrays in <tt>regression</tt> als x, die zweite Spalte des Arrays als y.</li>
    <li>Plotte y über x in einem Scatterplot mit entsprechenden Achsenbeschriftungen und einem Titel.</li>
    <li>Berechne den Korrelationskoeffizienten zwischen x und y und gib diesen formattiert aus.</li>
    <li>Berechne die lineare Regression 1. bis 12. Ordnung.</li>
    <li>Generiere 12 (Sub-)Plots, die sowohl die Daten in x und y als Scatterplot, als auch jeweils eine lineare Regression bestimmter Ordnung übersichtlich darstellen.</li>
    <li>Interpretiere das Ergebnis und begründe als Kommentar oder in einer Markdownzelle, welche Regressionskurve aus deiner Sicht die Daten am Besten beschreibt.</li>
</ol>

In [ ]:
# regressionsdaten laden
# np.load() liest binäre .npy-dateien — numpy's eigenes speicherformat für arrays
regression = np.load('data02/regression.npy')

xData = regression[:, 0]
yData = regression[:, 1]

# erst mal die rohdaten ansehen bevor wir irgendetwas fitten
plt.figure(figsize=(7, 5))
plt.scatter(xData, yData, s=15, alpha=0.7)
plt.title('UE02.c) – Rohdaten aus regression.npy')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()

# pearson-korrelation zwischen x und y
nReg = len(xData)
sXY = np.sum((xData - xData.mean()) * (yData - yData.mean())) / (nReg - 1)
corrCoefReg = sXY / (xData.std(ddof=1) * yData.std(ddof=1))
print(f'Korrelationskoeffizient r(x, y) = {corrCoefReg:.4f}')

# polynome vom grad 1 bis 12 fitten
# ein polynom grad i ist trotzdem "lineare regression" — linear in seinen i+1 koeffizienten
# Polynomial.fit findet diese koeffizienten durch minimierung der quadratischen residuen
#
# np.linspace(start, stop, num) erzeugt num gleichmäßig verteilte werte zwischen start und stop
# — wird hier für eine glatte kurve verwendet, nicht für die datenpunkte selbst
xLine = np.linspace(xData.min(), xData.max(), 500)

# plt.subplots(rows, cols) erstellt eine figure mit rows×cols achsen auf einmal
# — gibt (fig, axes) zurück, wobei axes ein 2D-array der größe (rows, cols) ist
# axes.flatten() macht daraus ein 1D-array, damit man per axes[i] indexieren kann
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

for i in range(1, 13):
    ax = axes[i - 1]
    polyModel = Polynomial.fit(xData, yData, deg=i)

    ax.scatter(xData, yData, s=8, alpha=0.5, label='data')
    ax.plot(xLine, polyModel(xLine), color='red', linewidth=1.5)
    ax.set_title(f'Ordnung {i}', fontsize=10)
    ax.set_xlabel('x', fontsize=8)
    ax.set_ylabel('y', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('UE02.c) – Polynomielle Regression, Ordnung 1–12', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation UE02.c)**

Der Pearson-Korrelationskoeffizient r zwischen x und y liegt nahe 0 — obwohl die Daten offensichtlich einen Zusammenhang haben. Das zeigt eine grundlegende Einschränkung von r: er misst nur **lineare** Zusammenhänge. Die Daten folgen erkennbar einer nichtlinearen Kurve, weshalb ein Polynom höherer Ordnung viel besser passt als eine Gerade.

Visuell ist ab **Ordnung 3–5** eine deutliche Verbesserung erkennbar. Sehr hohe Ordnungen (ab ca. 9–10) beginnen zu **overfitting**: die Kurve folgt dem Rauschen in den Daten statt dem echten Trend und schwingt zwischen den Datenpunkten stark aus.

Rein visuell erscheint **Ordnung 4 oder 5** als guter Kompromiss — die Kurve folgt dem Trend ohne zu überschwingen.

## UE02.d)
Erweiterung von UE02.c) um die Berechnung der Summe der Residuenquadrate $SQR$ <i>(engl. least squares residuals, $RSS$)</i> und die Berechnung des korrigierten Bestimmtheitsmaßes

$\bar{r}_{xy}^{2} = \cfrac{\hat{s}^2}{s_y^2}= 1 - \cfrac{\cfrac{1}{n-k}\;\sum_{i=1}^{n}\;(y_i - \hat{y}_i)^2}{\cfrac{1}{n-1}\;\sum_{i=1}^{n}\;(y_i - \bar{y}_i)^2}$

<ol>
    <li>Erweitere UE02.c) so, dass für alle Regressionskurven die Summe der Residuenquadrate im (Sub-)Plot dargestellt wird.<br>
        <strong>Tipp:</strong> Parameter <tt>full=True</tt> beim Aufruf der Methode <a href="https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.Polynomial.fit.html">Polynomial.fit()</a> setzen und das return value <tt>resid</tt> verwenden.
    </li>
    <li>Interpretiere das Ergebnis und nimm dabei speziell Bezug auf deine Erkenntnisse aus UE02.c). Deckt sich der Eindruck des visuellen "best fit" mit dem numerischen Ergebnis?</li>
    <li>Zuletzt ist das korrigierte Bestimmtheitsmaß $\bar{r}_{xy}^{2}$ für alle Regressionskurven zu rechnen und ebenfalls in allen (Sub-)Plots darzustellen.</li>
    <li>Entscheide nun final anhand der korrigierten Bestimmtheitsmaße, welche Regressionskurve einen adäquaten Kompromiss zwischen Modellkomplexität und fit auf die Daten bietet.</li>
</ol>

In [ ]:
# c) erweitern um RSS und korrigiertes bestimmtheitsmas
#
# RSS: Σ(yᵢ − ŷᵢ)² — reiner fit-fehler, fällt immer mit steigender ordnung
#
# das korrigierte r² bestraft unnötige modellkomplexität:
#   r̄²_xy = 1 − [1/(n−i) · Σ(yᵢ − ŷᵢ)²] / [1/(n−1) · Σ(yᵢ − ȳ)²]
# i = anzahl der parameter (= grad + 1 beim polynom)
# wenn neue parameter den fit nicht wirklich verbessern, steigt r̄² nicht mehr

nReg = len(xData)
xLine = np.linspace(xData.min(), xData.max(), 500)

# gesamtvarianz in y — für alle modelle gleich, einmal berechnen
totalVar = np.sum((yData - yData.mean()) ** 2) / (nReg - 1)

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

for i in range(1, 13):
    ax = axes[i - 1]

    # Polynomial.fit(..., full=True) gibt zusätzlich zum modell auch die rohen residuensummen zurück
    # rückgabewert: (modell, [resid, rank, sv, rcond])
    # resid[0] = Σ(yᵢ − ŷᵢ)² — aber nur wenn n > grad+1, sonst leeres array (exakter fit)
    polyModel, (resid, *_) = Polynomial.fit(xData, yData, deg=i, full=True)
    yHat = polyModel(xData)

    # numpy füllt resid nur wenn n > grad+1 (sonst ist ein exakter fit möglich)
    if len(resid) > 0:
        rss = resid[0]
    else:
        rss = np.sum((yData - yHat) ** 2)

    # korrigiertes r²: belohnt besseren fit, bestraft mehr parameter
    kParams = i + 1
    residVar = np.sum((yData - yHat) ** 2) / (nReg - kParams)
    adjRSq = 1 - residVar / totalVar

    ax.scatter(xData, yData, s=8, alpha=0.5)
    ax.plot(xLine, polyModel(xLine), color='red', linewidth=1.5)
    ax.set_title(f'Ordnung {i}', fontsize=10)
    ax.set_xlabel('x', fontsize=8)
    ax.set_ylabel('y', fontsize=8)
    ax.tick_params(labelsize=7)

    # ax.text(x, y, text) setzt einen text an position (x, y) im plot
    # transform=ax.transAxes bedeutet: koordinaten relativ zur achse (0,0 = unten-links, 1,1 = oben-rechts)
    # bbox=dict(...) zeichnet eine box um den text — boxstyle, farbe und transparenz einstellbar
    ax.text(0.05, 0.95, f'RSS = {rss:.2f}\nR²adj = {adjRSq:.4f}',
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='purple', alpha=0.3))

plt.suptitle('UE02.d) – RSS und korrigiertes Bestimmtheitsmaß r̄²', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation UE02.d)**

**RSS (Summe der Residuenquadrate):** Fällt monoton mit steigender Ordnung — ein Modell mit mehr Parametern kann die Datenpunkte immer besser interpolieren. Allein aus dem RSS kann man nicht ableiten, welches Modell *generalisiert*.

**Korrigiertes Bestimmtheitsmaß $\bar{r}^2$:** Hier wird Modellkomplexität bestraft. Der Wert steigt zunächst stark an (Ordnung 1→3), flacht dann ab und kann bei sehr hohen Ordnungen wieder sinken — weil der Gewinn im Fit die Strafe für die zusätzlichen Parameter nicht mehr aufwiegt.

**Fazit:** Das korrigierte $\bar{r}^2$ bestätigt den visuellen Eindruck aus c): Die optimale Ordnung liegt bei **4 oder 5**. Ab da bringen weitere Parameter keine nennenswerte Verbesserung mehr, erhöhen aber die Komplexität — und damit das Risiko von Overfitting auf neue Daten.

Das numerische Ergebnis deckt sich mit dem visuellen "best fit" aus c) und liefert damit eine objektive Grundlage für die Modellwahl.